In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
history = pd.read_csv(
    "../data/processed/asset_history_simulated.csv"
)

print(
    "Asset history rows:",
    len(history)
)

history.head()

Asset history rows: 92000


,asset_id,section_id,department,asset_type,snapshot_date,asset_age_years,condition_score,criticality,usage_factor,weather_stress
0,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-01-01,8,93.126212,6,0.565548,0.416271
1,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-02-01,8,92.456850,6,0.565548,0.087173
2,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-03-01,8,95.602455,6,0.565548,0.899679
3,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-04-01,8,96.610857,6,0.565548,0.408667
4,ASSET00001,RTM-VAD-01,TRACTION,OHE,2019-05-01,8,94.519420,6,0.565548,0.391072


In [3]:
print(
    history.columns.tolist()
)

['asset_id', 'section_id', 'department', 'asset_type', 'snapshot_date', 'asset_age_years', 'condition_score', 'criticality', 'usage_factor', 'weather_stress']


In [4]:
date_candidates = [
    column
    for column in history.columns
    if "date" in column.lower()
    or "time" in column.lower()
]

print(
    "Possible date columns:",
    date_candidates
)

Possible date columns: ['snapshot_date']


In [5]:
history["snapshot_date"] = pd.to_datetime(
    history["snapshot_date"],
    errors="coerce"
)

In [6]:
history = history.sort_values(
    [
        "asset_id",
        "snapshot_date"
    ]
).reset_index(drop=True)

print(
    "History sorted successfully."
)

History sorted successfully.


In [7]:
anomaly_features = [
    "condition_score",
    "usage_factor",
    "weather_stress",
    "criticality",
    "asset_age_years"
]

print(
    "Anomaly features:"
)

print(
    anomaly_features
)

Anomaly features:
['condition_score', 'usage_factor', 'weather_stress', 'criticality', 'asset_age_years']


In [8]:
history[
    anomaly_features
].isna().sum()

condition_score    0
usage_factor       0
weather_stress     0
criticality        0
asset_age_years    0
dtype: int64

In [9]:
X_anomaly = history[
    anomaly_features
].copy()

X_anomaly = X_anomaly.apply(
    pd.to_numeric,
    errors="coerce"
)

print(
    "Feature matrix shape:",
    X_anomaly.shape
)

Feature matrix shape: (92000, 5)


In [10]:
imputer = SimpleImputer(
    strategy="median"
)

X_anomaly_imputed = imputer.fit_transform(
    X_anomaly
)

print(
    "Missing values handled."
)

Missing values handled.


In [11]:
scaler = StandardScaler()

X_anomaly_scaled = scaler.fit_transform(
    X_anomaly_imputed
)

print(
    "Features standardized."
)

Features standardized.


In [12]:
anomaly_model = IsolationForest(
    n_estimators=300,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

anomaly_model.fit(
    X_anomaly_scaled
)

print(
    "Isolation Forest trained."
)

Isolation Forest trained.


In [13]:
history[
    "anomaly_label"
] = anomaly_model.predict(
    X_anomaly_scaled
)

In [14]:
history[
    "is_anomaly"
] = (
    history[
        "anomaly_label"
    ]
    == -1
).astype(int)

print(
    "Anomalies detected:",
    history["is_anomaly"].sum()
)

Anomalies detected: 1840


In [15]:
history[
    "anomaly_score"
] = -anomaly_model.score_samples(
    X_anomaly_scaled
)

history[
    [
        "asset_id",
        "condition_score",
        "usage_factor",
        "weather_stress",
        "anomaly_score",
        "is_anomaly"
    ]
].head(20)

,asset_id,condition_score,usage_factor,weather_stress,anomaly_score,is_anomaly
0,ASSET00001,93.126212,0.565548,0.416271,0.512116,0
1,ASSET00001,92.456850,0.565548,0.087173,0.535786,0
2,ASSET00001,95.602455,0.565548,0.899679,0.549397,0
3,ASSET00001,96.610857,0.565548,0.408667,0.530791,0
4,ASSET00001,94.519420,0.565548,0.391072,0.522430,0
5,ASSET00001,94.714184,0.565548,0.639993,0.520906,0
6,ASSET00001,98.417893,0.565548,0.765228,0.552369,0
7,ASSET00001,91.194071,0.565548,0.975113,0.547050,0
8,ASSET00001,92.403716,0.565548,0.502687,0.508989,0
9,ASSET00001,94.520897,0.565548,0.425128,0.519152,0


In [16]:
history[
    "anomaly_score"
].describe()

count    92000.000000
mean         0.520340
std          0.033435
min          0.445972
25%          0.495793
50%          0.518199
75%          0.542009
max          0.671790
Name: anomaly_score, dtype: float64

In [17]:
threshold_95 = history[
    "anomaly_score"
].quantile(0.95)

threshold_99 = history[
    "anomaly_score"
].quantile(0.99)

print(
    "95th percentile:",
    threshold_95
)

print(
    "99th percentile:",
    threshold_99
)

95th percentile: 0.5790568143385831
99th percentile: 0.6066227078265451


In [18]:
history[
    "anomaly_category"
] = np.select(
    [
        history["anomaly_score"] >= threshold_99,
        history["anomaly_score"] >= threshold_95
    ],
    [
        "CRITICAL",
        "SUSPICIOUS"
    ],
    default="NORMAL"
)

history[
    "anomaly_category"
].value_counts()

anomaly_category
NORMAL        87400
SUSPICIOUS     3680
CRITICAL        920
Name: count, dtype: int64

In [19]:
top_anomalies = (
    history[
        history["is_anomaly"] == 1
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
)

top_anomalies[
    [
        "asset_id",
        "section_id",
        "condition_score",
        "usage_factor",
        "weather_stress",
        "criticality",
        "anomaly_score",
        "anomaly_category"
    ]
].head(20)

,asset_id,section_id,condition_score,usage_factor,weather_stress,criticality,anomaly_score,anomaly_category
21061,ASSET00229,RTM-VAD-01,56.293564,0.565548,0.199921,5,0.671790,CRITICAL
12594,ASSET00137,JHS-BINA-01,54.274236,0.662164,0.968872,10,0.667691,CRITICAL
56756,ASSET00617,NDL-MTJ-01,61.458097,1.424733,0.932858,5,0.665359,CRITICAL
12574,ASSET00137,JHS-BINA-01,57.052836,0.662164,0.015116,10,0.664358,CRITICAL
48648,ASSET00529,RTM-VAD-01,54.843879,0.565548,0.054902,10,0.661661,CRITICAL
21050,ASSET00229,RTM-VAD-01,58.262524,0.565548,0.157188,5,0.661456,CRITICAL
21056,ASSET00229,RTM-VAD-01,58.867098,0.565548,0.131507,5,0.661182,CRITICAL
12597,ASSET00137,JHS-BINA-01,59.604551,0.662164,0.913454,10,0.660046,CRITICAL
76634,ASSET00833,BINA-BPL-01,58.635645,0.570051,0.092463,9,0.658504,CRITICAL
72586,ASSET00789,NDL-MTJ-01,56.130828,1.424733,0.821234,9,0.657605,CRITICAL


In [20]:
if "department" in history.columns:

    department_anomalies = (
        history
        .groupby("department")
        .agg(
            total_observations=(
                "asset_id",
                "count"
            ),
            anomalies=(
                "is_anomaly",
                "sum"
            ),
            average_anomaly_score=(
                "anomaly_score",
                "mean"
            )
        )
        .reset_index()
    )

    department_anomalies[
        "anomaly_rate"
    ] = (
        department_anomalies[
            "anomalies"
        ]
        /
        department_anomalies[
            "total_observations"
        ]
    )

    display(
        department_anomalies
    )

else:

    print(
        "Department column not available."
    )

,department,total_observations,anomalies,average_anomaly_score,anomaly_rate
0,ENGINEERING,32476,663,0.521880,0.020415
1,S&T,31004,631,0.519739,0.020352
2,TRACTION,28520,546,0.519239,0.019144


In [21]:
section_anomalies = (
    history
    .groupby("section_id")
    .agg(
        observations=(
            "asset_id",
            "count"
        ),
        anomalies=(
            "is_anomaly",
            "sum"
        ),
        average_score=(
            "anomaly_score",
            "mean"
        ),
        maximum_score=(
            "anomaly_score",
            "max"
        )
    )
    .reset_index()
)

section_anomalies[
    "anomaly_rate"
] = (
    section_anomalies["anomalies"]
    /
    section_anomalies["observations"]
)

section_anomalies = (
    section_anomalies
    .sort_values(
        "anomaly_rate",
        ascending=False
    )
)

section_anomalies

,section_id,observations,anomalies,average_score,maximum_score,anomaly_rate
2,BPL-RTM-01,9292,477,0.549433,0.651274,0.051334
6,NDL-MTJ-01,9476,321,0.540619,0.665359,0.033875
4,JHS-BINA-01,8648,220,0.527149,0.667691,0.025439
7,RTM-VAD-01,9568,239,0.529552,0.671790,0.024979
1,BINA-BPL-01,9660,215,0.524780,0.658504,0.022257
5,MTJ-AGC-01,8832,143,0.518081,0.654173,0.016191
9,VAD-SRT-01,10028,78,0.506569,0.653330,0.007778
0,AGC-GWL-01,8924,65,0.507675,0.655257,0.007284
3,GWL-JHS-01,9200,44,0.496945,0.650282,0.004783
8,SRT-MUM-01,8372,38,0.500495,0.639904,0.004539


In [22]:
score_min = history[
    "anomaly_score"
].min()

score_max = history[
    "anomaly_score"
].max()

history[
    "anomaly_risk"
] = (
    history["anomaly_score"]
    - score_min
) / (
    score_max
    - score_min
    + 1e-9
)

history[
    "anomaly_risk"
] = history[
    "anomaly_risk"
].clip(0, 1)

In [23]:
history[
    "condition_risk"
] = (
    100
    - history["condition_score"]
) / 100

history[
    "criticality_risk"
] = (
    history["criticality"]
    / 10
).clip(0, 1)

history[
    "anomaly_enhanced_risk"
] = (
    0.50
    * history["anomaly_risk"]
    +
    0.30
    * history["condition_risk"]
    +
    0.20
    * history["criticality_risk"]
)

history[
    "anomaly_enhanced_risk"
] = history[
    "anomaly_enhanced_risk"
].clip(0, 1)

In [24]:
anomaly_priority_assets = (
    history
    .sort_values(
        "anomaly_enhanced_risk",
        ascending=False
    )
)

anomaly_priority_assets[
    [
        "asset_id",
        "section_id",
        "condition_score",
        "criticality",
        "anomaly_score",
        "anomaly_risk",
        "anomaly_enhanced_risk"
    ]
].head(20)

,asset_id,section_id,condition_score,criticality,anomaly_score,anomaly_risk,anomaly_enhanced_risk
12594,ASSET00137,JHS-BINA-01,54.274236,10,0.667691,0.981850,0.828102
48648,ASSET00529,RTM-VAD-01,54.843879,10,0.661661,0.955146,0.813041
12574,ASSET00137,JHS-BINA-01,57.052836,10,0.664358,0.967089,0.812386
12597,ASSET00137,JHS-BINA-01,59.604551,10,0.660046,0.947995,0.795184
77821,ASSET00846,MTJ-AGC-01,55.526996,10,0.654173,0.921986,0.794412
48652,ASSET00529,RTM-VAD-01,57.891667,10,0.655939,0.929805,0.791228
86934,ASSET00945,JHS-BINA-01,57.931663,10,0.655744,0.928942,0.790676
48664,ASSET00529,RTM-VAD-01,58.700023,10,0.655066,0.925942,0.786871
12573,ASSET00137,JHS-BINA-01,60.169423,10,0.656852,0.933849,0.786416
72586,ASSET00789,NDL-MTJ-01,56.130828,9,0.657605,0.937182,0.780199


In [25]:
latest_anomaly_state = (
    history
    .sort_values("snapshot_date")
    .groupby("asset_id")
    .tail(1)
    .copy()
)

print(
    "Latest asset states:",
    len(latest_anomaly_state)
)

Latest asset states: 1000


In [26]:
anomaly_output = latest_anomaly_state[
    [
        "asset_id",
        "section_id",
        "condition_score",
        "usage_factor",
        "weather_stress",
        "criticality",
        "anomaly_score",
        "is_anomaly",
        "anomaly_category",
        "anomaly_risk",
        "anomaly_enhanced_risk"
    ]
].copy()

anomaly_output.head(20)

,asset_id,section_id,condition_score,usage_factor,weather_stress,criticality,anomaly_score,is_anomaly,anomaly_category,anomaly_risk,anomaly_enhanced_risk
91631,ASSET00996,JHS-BINA-01,65.464923,0.662164,0.468746,10,0.547541,0,NORMAL,0.449781,0.528496
91079,ASSET00990,NDL-MTJ-01,78.644027,1.424733,0.130947,10,0.570725,0,NORMAL,0.552450,0.540293
91447,ASSET00994,BINA-BPL-01,68.510339,0.570051,0.716770,10,0.552556,0,NORMAL,0.471992,0.530465
91355,ASSET00993,AGC-GWL-01,55.722818,1.024529,0.763244,9,0.620089,1,CRITICAL,0.771048,0.698356
91907,ASSET00999,BINA-BPL-01,59.581701,0.570051,0.824726,5,0.629734,1,CRITICAL,0.813763,0.628136
91263,ASSET00992,BINA-BPL-01,82.687283,0.570051,0.766261,8,0.503340,0,NORMAL,0.254045,0.338961
91815,ASSET00998,SRT-MUM-01,89.348848,1.118228,0.295727,5,0.496053,0,NORMAL,0.221776,0.242841
91539,ASSET00995,AGC-GWL-01,85.746137,1.024529,0.143018,7,0.483494,0,NORMAL,0.166162,0.265843
91171,ASSET00991,VAD-SRT-01,76.251754,1.141546,0.821106,10,0.492695,0,NORMAL,0.206904,0.374697
91723,ASSET00997,GWL-JHS-01,75.788657,1.118105,0.633837,10,0.489105,0,NORMAL,0.191007,0.368137


In [27]:
anomaly_output.to_csv(
    "../data/predictions/anomaly_predictions.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/anomaly_predictions.csv"
)

Saved: ../data/predictions/anomaly_predictions.csv


In [28]:
print(
    "===== ANOMALY DETECTION SUMMARY ====="
)

print(
    "Total historical observations:",
    len(history)
)

print(
    "Historical anomalies:",
    history["is_anomaly"].sum()
)

print(
    "Historical anomaly rate:",
    round(
        history["is_anomaly"].mean(),
        4
    )
)

print(
    "\nLatest asset states:",
    len(anomaly_output)
)

print(
    "\nLatest anomaly categories:"
)

print(
    anomaly_output[
        "anomaly_category"
    ].value_counts()
)

print(
    "\nHighest-risk assets:"
)

display(
    anomaly_output[
        [
            "asset_id",
            "section_id",
            "anomaly_score",
            "anomaly_category",
            "anomaly_enhanced_risk"
        ]
    ]
    .sort_values(
        "anomaly_enhanced_risk",
        ascending=False
    )
    .head(15)
)


===== ANOMALY DETECTION SUMMARY =====
Total historical observations: 92000
Historical anomalies: 1840
Historical anomaly rate: 0.02

Latest asset states: 1000

Latest anomaly categories:
anomaly_category
NORMAL        886
SUSPICIOUS     76
CRITICAL       38
Name: count, dtype: int64

Highest-risk assets:


,asset_id,section_id,anomaly_score,anomaly_category,anomaly_enhanced_risk
48667,ASSET00529,RTM-VAD-01,0.640458,CRITICAL,0.750509
76635,ASSET00833,BINA-BPL-01,0.641777,CRITICAL,0.745692
26955,ASSET00293,NDL-MTJ-01,0.645157,CRITICAL,0.742939
8003,ASSET00087,BPL-RTM-01,0.636015,CRITICAL,0.735578
12603,ASSET00137,JHS-BINA-01,0.636290,CRITICAL,0.735137
21711,ASSET00236,MTJ-AGC-01,0.633561,CRITICAL,0.730610
67711,ASSET00736,BINA-BPL-01,0.640901,CRITICAL,0.721873
66515,ASSET00723,SRT-MUM-01,0.639904,CRITICAL,0.709281
44711,ASSET00486,NDL-MTJ-01,0.635034,CRITICAL,0.704021
18031,ASSET00196,AGC-GWL-01,0.622579,CRITICAL,0.703884
